In [37]:
import os
import re
import logging
from datetime import datetime, timedelta
import time
from tqdm import tqdm
import pandas as pd
from dataclasses import dataclass, fields, is_dataclass
from __future__ import annotations
from typing import Any, Dict, List, Tuple, get_args, get_origin, Union
import json

from params.paths import ROOT_DIR
from api_requests.meeting_convo_collector import MeetingConvoCollector
from file_handling.file_read_writer import read_json, write_json, create_dir, write_file

OUTPUT_DIR = os.path.join(ROOT_DIR, 'data', 'data_all_speeches')
create_dir(OUTPUT_DIR)

In [38]:
@dataclass(frozen=True)
class SpeechRecord(BaseModel):
	speaker: str
	speech: str
	speechID: str
	speechOrder: int
	speakerYomi: str
	speakerGroup: str
	speakerPosition: str
	speakerRole: str
	speechURL: str
	startPage: int
	createTime: str
	updateTime: str

@dataclass(frozen=True)
class MeetingRecord(BaseModel):
	issueID:str
	imageKind: str
	searchObject: int
	session: int
	meetingURL:str
	nameOfHouse: str
	nameOfMeeting: str
	issue: str
	date: str
	closing: str
	speechRecord: List[SpeechRecord]
	pdfURL:str

@dataclass(frozen=True)
class FetchedResponse(BaseModel):
	numberOfRecords:int
	numberOfReturn: int
	startRecord: int
	nextRecordPosition: int
	meetingRecord: List[MeetingRecord]

NameError: name 'BaseModel' is not defined

In [ ]:
mcc = MeetingConvoCollector("https://kokkai.ndl.go.jp/api/meeting?")


starting_point = 1
fetch_num_per_request = 10

# --- Generic recursive constructor ---

def from_dict(cls, data):
    if data is None:
        return None

    if is_dataclass(cls):
        kwargs = {}
        for f in fields(cls):
            raw = data.get(f.name, None)
            kwargs[f.name] = _convert_value_with_defaults(f.type, raw)
        return cls(**kwargs)

    origin = get_origin(cls)
    if origin in (list, List):
        (elem_type,) = get_args(cls)
        # --- KEY FIX: accept dict-as-singleton for one-or-many APIs ---
        if data is None:
            return []
        if isinstance(data, dict):
            data = [data]
        return [ _convert_value_with_defaults(elem_type, x) for x in data ]

    return _cast_primitive(cls, data)

def _convert_value_with_defaults(tp, val):
    origin = get_origin(tp)

    if origin is Union:  # Optional[T] etc.
        args = [a for a in get_args(tp) if a is not type(None)]
        if val is None:
            return None
        return _convert_value_with_defaults(args[0], val)

    if is_dataclass(tp):
        return from_dict(tp, val or {})

    if origin in (list, List):
        (elem_type,) = get_args(tp)
        if val is None:
            return []
        # --- SAME FIX applied here too for nested lists like speechRecord ---
        if isinstance(val, dict):
            val = [val]
        return [ _convert_value_with_defaults(elem_type, x) for x in val ]

    return _cast_primitive(tp, val)

def _cast_primitive(tp, val):
    if val is None:
        return None
    if tp in (int, float, str, bool):
        try:
            return tp(val)
        except Exception:
            return val
    return val




while True:
	conditions_list = [
		f"any=''",
		f"recordPacking=json",
		f"startRecord={starting_point}",
		f"maximumRecords={fetch_num_per_request}"
	]
	response, next_position = mcc.make_one_request(conditions_list, starting_point)
	response = from_dict(FetchedResponse, response)

	if next_position is None:
		break
	starting_point = next_position
	meetingRecords = response.meetingRecord
	for mr in meetingRecords:
		mr = MeetingRecord(**mr)
		issueID = mr.issueID
		imageKind = mr.imageKind
		searchObject = mr.searchObject
		session = mr.session
		nameOfHouse = mr.nameOfHouse
		nameOfMeeting = mr.nameOfMeeting
		issue = mr.issue
		os.makedirs(os.path.join(OUTPUT_DIR, issueID), exist_ok=True)
		with open(os.path.join(OUTPUT_DIR, issueID, f"meta.json"), "w", encoding="utf-8") as f:
			json.dump({
				"issueID": issueID,
				"imageKind": imageKind,
				"searchObject": searchObject,
				"session": session,
				"nameOfHouse": nameOfHouse,
				"nameOfMeeting": nameOfMeeting,
				"issue": issue
			}, f, ensure_ascii=False, indent=4)

		with open(os.path.join(OUTPUT_DIR, issueID, f"speeches.jsonl"), "w", encoding="utf-8") as f:
			for speech in mr.speechRecord:
				json.dump(speech, f, ensure_ascii=False)
				f.write("\n")





https://kokkai.ndl.go.jp/api/meeting?startRecord=1&any=''&recordPacking=json&startRecord=1&maximumRecords=10
https://kokkai.ndl.go.jp/api/meeting?startRecord=11&any=''&recordPacking=json&startRecord=11&maximumRecords=10
https://kokkai.ndl.go.jp/api/meeting?startRecord=21&any=''&recordPacking=json&startRecord=21&maximumRecords=10


KeyboardInterrupt: 